# 02 — Feature Engineering

Builds the full feature matrix used for ML/DL models.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
ROOT = Path('..')

## 1. Build feature matrix

In [ ]:
from src.data.fetch_data import fetch_entsoe_load, fetch_openmeteo_weather
from src.data.preprocess import build_features, train_test_split_temporal

load    = fetch_entsoe_load()
weather = fetch_openmeteo_weather()

df = build_features(load, weather)
print(f'Feature matrix: {df.shape}')
df.head(3)

## 2. Feature statistics and null check

In [ ]:
print('Null counts:', df.isna().sum().sum())
df.describe().T.sort_values('std', ascending=False).head(20)

## 3. Correlation heatmap of lag features

In [ ]:
lag_cols = [c for c in df.columns if c.startswith('lag_')]
corr = df[['load_MW'] + lag_cols].corr()[['load_MW']].sort_values('load_MW', ascending=False)

plt.figure(figsize=(6, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation of Lag Features with Target')
plt.tight_layout()
plt.show()

## 4. Train / test split

In [ ]:
train, test = train_test_split_temporal(df, test_ratio=0.15)
print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Train ends : {train.index[-1]}')
print(f'Test starts: {test.index[0]}')

## 5. Save processed dataset

In [ ]:
# Already saved by build_features() — just confirm
out = ROOT / 'data' / 'processed' / 'features.parquet'
print(f'Saved at: {out}  (exists: {out.exists()})')